In [5]:
# Fix event_id in 2024_small_normalized.csv and 2025_small_normalized.csv (IN PLACE)
# - Uses your authoritative schedule mapping (season + event_name -> event_id)
# - Robust to small event_name differences via normalization + fuzzy matching
# - Produces a change report + unmatched list

from __future__ import annotations

from pathlib import Path
import re
import pandas as pd
from difflib import SequenceMatcher

# =========================
# INPUT FILES (IN PLACE)
# =========================
FILES = [
    Path("/Data/Clean/Leagues/2026_small.csv"),
    # Path("/Users/joshmacbook/python_projects/OAD/Data/Clean/Leagues/2025_small_normalized.csv"),
]

# =========================
# YOUR AUTHORITATIVE SCHEDULE
# =========================
# Paste exactly as provided (I’m encoding it as rows)
SCHEDULE_ROWS = [
    ("Sony Open in Hawaii", 6, 2026, 1),
    ("The American Express", 2, 2026, 2),
    ("Farmers Insurance Open", 4, 2026, 3),
    ("WM Phoenix Open", 3, 2026, 4),
    ("AT&T Pebble Beach Pro-Am", 5, 2026, 5),
    ("The Genesis Invitational", 7, 2026, 6),
    ("Cognizant Classic", 10, 2026, 7),
    ("Arnold Palmer Invitational presented by Mastercard", 9, 2026, 8),
    ("THE PLAYERS Championship", 11, 2026, 9),
    ("Valspar Championship", 475, 2026, 10),
    ("Texas Children's Houston Open", 20, 2026, 11),
    ("Valero Texas Open", 41, 2026, 12),
    ("Masters Tournament", 14, 2026, 13),
    ("RBC Heritage", 12, 2026, 14),
    ("Cadillac Championship", None, 2026, 15),  # event_id not provided
    ("Truist Championship", 480, 2026, 16),
    ("PGA Championship", 33, 2026, 17),
    ("THE CJ CUP Byron Nelson", 19, 2026, 18),
    ("Charles Schwab Challenge", 21, 2026, 19),
    ("the Memorial Tournament presented by Workday", 23, 2026, 20),
    ("RBC Canadian Open", 32, 2026, 21),
    ("U.S. Open", 26, 2026, 22),
    ("Travelers Championship", 34, 2026, 23),
    ("John Deere Classic", 30, 2026, 24),
    ("Genesis Scottish Open", 541, 2026, 25),
    ("The Open Championship", 100, 2026, 26),
    ("3M Open", 525, 2026, 27),
    ("Rocket Classic", 524, 2026, 28),
    ("Wyndham Championship", 13, 2026, 29),
    ("FedEx St. Jude Championship", 27, 2026, 30),
    ("BMW Championship", 28, 2026, 31),
]


sched = pd.DataFrame(SCHEDULE_ROWS, columns=["event_name", "event_id", "season", "week_num"])

# =========================
# NORMALIZATION + MATCHING
# =========================
STOPWORDS = {
    "the", "a", "an",
    "presented", "by",
    "in", "at", "of", "on", "for",
    "championship", "tournament",  # optional; keep if needed
}

def normalize_event_name(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower().strip()

    # unify punctuation / special cases
    s = s.replace("&", "and")
    s = s.replace("u.s.", "us")
    s = s.replace("st.", "st")
    s = re.sub(r"[^a-z0-9\s]", " ", s)       # remove punctuation
    s = re.sub(r"\s+", " ", s).strip()

    toks = [t for t in s.split(" ") if t and t not in STOPWORDS]
    return " ".join(toks)

sched["event_name_norm"] = sched["event_name"].map(normalize_event_name)

# Build lookup per season
sched_by_season = {
    int(season): sdf.copy().reset_index(drop=True)
    for season, sdf in sched.groupby("season")
}

def best_fuzzy_match(name_norm: str, candidates: list[str]) -> tuple[int, float]:
    """
    Returns (best_index, best_score) where score is 0..1
    """
    best_i, best_score = -1, -1.0
    for i, cand in enumerate(candidates):
        score = SequenceMatcher(None, name_norm, cand).ratio()
        if score > best_score:
            best_score, best_i = score, i
    return best_i, best_score

def fix_event_ids_inplace(df: pd.DataFrame, min_score: float = 0.88) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      df_fixed, changes_df, unmatched_df
    """
    if "season" not in df.columns:
        raise ValueError("Expected column 'season' in file.")
    if "event_name" not in df.columns:
        raise ValueError("Expected column 'event_name' in file.")
    if "event_id" not in df.columns:
        raise ValueError("Expected column 'event_id' in file.")

    df = df.copy()
    df["event_name_norm"] = df["event_name"].map(normalize_event_name)

    changes = []
    unmatched = []

    # process each season separately (prevents cross-season wrong matches)
    for season_val, part_idx in df.groupby("season").groups.items():
        season_int = int(season_val)
        if season_int not in sched_by_season:
            unmatched.append({
                "season": season_int,
                "event_name": None,
                "event_name_norm": None,
                "reason": f"season {season_int} not present in schedule map",
            })
            continue

        sref = sched_by_season[season_int]
        cand_norms = sref["event_name_norm"].tolist()

        # try exact normalized match first
        part = df.loc[part_idx]
        exact_map = dict(zip(sref["event_name_norm"], sref["event_id"]))

        for row_i, row in part.iterrows():
            cur_event_id = row["event_id"]
            nm = row["event_name"]
            nm_norm = row["event_name_norm"]

            new_event_id = None
            match_type = None
            score = None
            matched_to = None

            if nm_norm in exact_map:
                new_event_id = int(exact_map[nm_norm])
                match_type = "exact_norm"
                score = 1.0
                matched_to = sref.loc[sref["event_name_norm"] == nm_norm, "event_name"].iloc[0]
            else:
                # fuzzy match
                best_i, best_score = best_fuzzy_match(nm_norm, cand_norms)
                if best_i >= 0 and best_score >= min_score:
                    new_event_id = int(sref.iloc[best_i]["event_id"])
                    match_type = "fuzzy"
                    score = float(best_score)
                    matched_to = str(sref.iloc[best_i]["event_name"])
                else:
                    unmatched.append({
                        "season": season_int,
                        "event_name": nm,
                        "event_name_norm": nm_norm,
                        "best_score": float(best_score) if best_i >= 0 else None,
                        "best_match_event_name": str(sref.iloc[best_i]["event_name"]) if best_i >= 0 else None,
                        "best_match_event_id": int(sref.iloc[best_i]["event_id"]) if best_i >= 0 else None,
                        "reason": f"no match >= {min_score}",
                    })
                    continue

            # apply change if needed
            if new_event_id is not None and (pd.isna(cur_event_id) or int(cur_event_id) != int(new_event_id)):
                df.at[row_i, "event_id"] = int(new_event_id)
                changes.append({
                    "season": season_int,
                    "event_name": nm,
                    "event_id_old": cur_event_id,
                    "event_id_new": new_event_id,
                    "match_type": match_type,
                    "score": score,
                    "matched_to_event_name": matched_to,
                })

    changes_df = pd.DataFrame(changes).sort_values(["season", "event_id_new", "event_name"], ignore_index=True) if changes else pd.DataFrame()
    unmatched_df = pd.DataFrame(unmatched).sort_values(["season", "event_name"], ignore_index=True) if unmatched else pd.DataFrame()

    # cleanup helper col
    df = df.drop(columns=["event_name_norm"], errors="ignore")

    return df, changes_df, unmatched_df


# =========================
# RUN
# =========================
for path in FILES:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df0 = pd.read_csv(path)

    df_fixed, changes_df, unmatched_df = fix_event_ids_inplace(df0, min_score=0.88)

    # Summary
    print("\n" + "="*90)
    print(f"FILE: {path}")
    print(f"Rows: {len(df0):,}")
    print(f"Changed rows: {len(changes_df):,}")
    print(f"Unmatched rows: {len(unmatched_df):,}")

    # Show samples
    if len(changes_df):
        print("\n--- CHANGES (first 25) ---")
        display(changes_df.head(25))

    if len(unmatched_df):
        print("\n--- UNMATCHED (review these) ---")
        display(unmatched_df)

        # If you want to fail hard when any unmatched exist, uncomment:
        # raise ValueError(f"Unmatched events exist for {path}. Review unmatched_df above.")

    # Write back IN PLACE (same filename)
    df_fixed.to_csv(path, index=False)
    print("✅ Wrote updated file IN PLACE.")


ValueError: Expected column 'event_id' in file.